# Predicción horaria con tres torres semánticas GRU-D

Este notebook entrena tres torres multivariables con atención temporal. Sus resúmenes se combinan con el contexto del paciente para predecir la sepsis por hora.

La configuración inicial usa todos los pacientes del conjunto A de entrenamiento y validación cruzada de tres folds. La prueba final está desactivada por defecto.

Antes de empezar, ejecuta `uv sync --extra model --group dev` en la raíz del proyecto, selecciona el kernel de `.venv` y comprueba que `cleaning.ipynb` haya exportado las particiones.

In [1]:
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
from tensorflow import keras

from appr_project.data import (
    PatientHours,
    Standardization,
    WindowBatches,
    load_partition,
    split_patients,
)
from appr_project.evaluation import predict_hours, summarize_metrics
from appr_project.models.semantic_grud import SEMANTIC_GROUPS_3
from appr_project.training import Hyperparameters, train_experiment

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
DATA_DIR = ROOT / "data" / "processed"


## 1. Configuración del piloto

El piloto usa 2.000 pacientes y una sola partición de validación. `source` admite `"A"`, `"B"` y `"A+B"`.

La búsqueda usa Optuna TPE con 10 ensayos iniciales, incluido el modelo base. La poda por mediana puede actuar desde el ensayo 6, al final de la época 8, con al menos 3 resultados de referencia en esa época. Para validar una configuración fija con 3 folds, usa `search="random"`, `trials=1`, `sample_patients=None` y `folds=3`.


In [ ]:
settings = dict(
    architecture="semantic_grud_3",
    source="A",
    sample_patients=4000,
    validation_fraction=0.20,
    folds=None,
    search="tpe",
    trials=1,
    epochs=30,
    patience=4,
    seed=2026,
)

base = Hyperparameters(
    window=8,
    units=8,
    dense_units=32,
    dropout=0.04527,
    learning_rate=0.000663334,
    batch_size=128,
    kernel_l2=0,
    recurrent_l2=0,
)


## 2. Inspeccionar la partición y un batch

Se muestran los pacientes, sus horas y cuántos tienen alguna etiqueta positiva. La fracción de horas positivas en validación sirve como referencia para el AP global y por origen. En CV, la vista previa corresponde al primer fold. La muestra y la partición se reproducen al entrenar con la misma semilla.

Cada entrada temporal tiene dimensiones `[batch, horas, variables, 4]`. Los cuatro canales son el valor estandarizado, la máscara de medición, el tiempo desde la medición previa y su valor. La máscara `valid_hours` marca las horas reales de la ventana.


In [36]:
frame = load_partition(DATA_DIR, "train", settings["source"])

train_ids, val_ids = next(split_patients(
    frame,
    validation_fraction=settings["validation_fraction"],
    folds=settings["folds"],
    seed=settings["seed"],
    sample_patients=settings["sample_patients"],
))

train_preview = frame.loc[frame.patient_id.isin(train_ids)]
val_preview = frame.loc[frame.patient_id.isin(val_ids)]

summary = []
for name, partition in (("train", train_preview), ("validation", val_preview)):
    patient_labels = partition.groupby("patient_id").SepsisLabel.max()
    summary.append({
        "partición": name,
        "pacientes": len(patient_labels),
        "pacientes positivos": int(patient_labels.sum()),
        "horas": len(partition),
    })
display(pd.DataFrame(summary).set_index("partición"))

validation_baseline = []
for source, rows in [("all", val_preview), *val_preview.groupby("source_set")]:
    positive_hours = rows.SepsisLabel.eq(1)
    validation_baseline.append({
        "source": source,
        "hours": len(rows),
        "positive_hours": int(positive_hours.sum()),
        "positive_hour_fraction": positive_hours.mean(),
    })

print("Referencia del AP en validación por hora")
display(pd.DataFrame(validation_baseline).set_index("source").round(4))


,pacientes,pacientes positivos,horas
partición,,,
train,3200,282,123809
validation,800,70,30575


Referencia del AP en validación por hora


,hours,positive_hours,positive_hour_fraction
source,,,
all,30575,672,0.022
A,30575,672,0.022


In [25]:
scaler = Standardization.fit(train_preview)
first_patient = train_preview.loc[train_preview.patient_id.eq(train_ids[0])]
hours = PatientHours(first_patient, scaler)
batches = WindowBatches(hours, base.window, base.batch_size)
inputs, labels = batches[0]

print("Formas de entrada:", {name: array.shape for name, array in inputs.items()})
print("Forma de las etiquetas:", labels.shape)
print("Paciente:", hours.metadata["patient_id"].iloc[0])
print("Horas válidas de la primera ventana:", inputs["valid_hours"][0].astype(int).tolist())
print("Etiqueta de la primera ventana:", int(labels[0, 0]))

for tower, variables in SEMANTIC_GROUPS_3.items():
    print(tower, list(variables))


Formas de entrada: {'temporal': (22, 8, 26, 4), 'valid_hours': (22, 8), 'static': (22, 10)}
Forma de las etiquetas: (22, 1)
Paciente: A_007783
Horas válidas de la primera ventana: [0, 0, 0, 0, 0, 0, 0, 1]
Etiqueta de la primera ventana: 0
hemodynamics_oxygenation ['HR', 'SBP', 'MAP', 'DBP', 'Lactate', 'Resp', 'O2Sat', 'SaO2', 'FiO2', 'PaCO2', 'pH', 'Temp']
renal_hepatic ['BUN', 'Creatinine', 'Calcium', 'Magnesium', 'Phosphate', 'Potassium', 'Glucose', 'AST', 'Alkalinephos', 'Bilirubin_total']
hematology ['Hct', 'Hgb', 'WBC', 'Platelets']


## 3. Entrenar y revisar los resultados

Cada ejecución crea una carpeta nueva en `runs/` con el modelo, la configuración, las particiones, las métricas y los tiempos. El prefijo `notebook-semantic-grud-3` identifica estas corridas.

Para comparar con `modeling_semantic.ipynb` (cinco torres), comprueba que las asignaciones de pacientes en `splits.csv` coincidan y usa el mismo origen de prueba. Una vez elegida la configuración de cada modelo, compara la precisión promedio (AP), la utilidad y el recall al umbral seleccionado en los archivos `test_<origen>_metrics.csv`.

In [37]:
run_dir = ROOT / "runs" / f"notebook-semantic-grud-3-{settings['source']}-{datetime.now():%Y%m%d-%H%M%S-%f}"
validation_metrics = train_experiment(frame, run_dir, base=base, **settings)

print(f"Resultados: {run_dir}")
display(validation_metrics.round(4))


[I 2026-11-16 18:07:00,068] A new study created in memory with name: no-name-71ae50a9-2283-40af-8a02-46e62897d96f


Trial 1 (1/1), fold 1/1: Hyperparameters(window=8, units=16, dense_units=64, dropout=0.12, learning_rate=0.007315693776479252, batch_size=128, use_hidden_dense=False, kernel_l2=0, recurrent_l2=0)
Epoch 1/30
968/968 - 43s - 44ms/step - loss: 0.0969 - pr_auc: 0.0757 - val_loss: 0.0971 - val_pr_auc: 0.0912
Epoch 2/30
968/968 - 22s - 23ms/step - loss: 0.0830 - pr_auc: 0.1969 - val_loss: 0.0959 - val_pr_auc: 0.1001
Epoch 3/30
968/968 - 21s - 22ms/step - loss: 0.0719 - pr_auc: 0.3334 - val_loss: 0.1155 - val_pr_auc: 0.0680
Epoch 4/30
968/968 - 23s - 24ms/step - loss: 0.0622 - pr_auc: 0.4560 - val_loss: 0.1187 - val_pr_auc: 0.0726
Epoch 5/30
968/968 - 21s - 21ms/step - loss: 0.0545 - pr_auc: 0.5553 - val_loss: 0.1372 - val_pr_auc: 0.0730
Epoch 6/30
968/968 - 20s - 21ms/step - loss: 0.0485 - pr_auc: 0.6304 - val_loss: 0.1455 - val_pr_auc: 0.0639
Resultados: c:\Users\palom\Documents\College\APPR\APPR_project\runs\notebook-semantic-grud-3-A-20261116-180659-553281


,source,patients,hours,positive_hours,positive_hour_fraction,average_precision,auroc,utility,alert_fraction,recall_at_threshold,threshold
0,all,800,30575,672,0.022,0.1007,0.786,0.4022,0.1915,0.6265,0.0227
1,A,800,30575,672,0.022,0.1007,0.786,0.4022,0.1915,0.6265,0.0227


In [14]:
display(pd.read_csv(run_dir / "trials.csv").round(4))

fold_metrics = pd.read_csv(run_dir / "fold_metrics.csv")
timing_columns = [
    "trial", "fold", "train_windows", "validation_windows",
    "preparation_seconds", "epochs_run", "seconds_per_epoch",
]
display(fold_metrics.loc[fold_metrics.source.eq("all"), timing_columns].round(2))


run_timing = json.loads((run_dir / "run_timing.json").read_text(encoding="utf-8"))
print(f"Tiempo total de ejecución: {run_timing['total_seconds']:.2f} s")

,trial,window,units,dense_units,dropout,learning_rate,batch_size,use_hidden_dense,kernel_l2,recurrent_l2,mean_validation_ap,std_validation_ap,state,train_windows,validation_windows,preparation_seconds,fit_seconds,epochs_run,best_epoch,seconds_per_epoch
0,1,8,16,64,0.0608,0.0073,128,False,0,0,0.1172,0.0,COMPLETE,123809,30575,2.3102,136.0164,5,1,27.2033


,trial,fold,train_windows,validation_windows,preparation_seconds,epochs_run,seconds_per_epoch
0,1,1,123809,30575,2.31,5,27.2


Tiempo total de ejecución: 144.99 s


## 4. Evaluación final opcional

Activa esta celda cuando hayas elegido la configuración definitiva. La utilidad y la proporción de horas con alerta en test se calculan con el umbral guardado en la corrida. Los resultados se muestran y se guardan por origen y de forma agregada. Usa `positive_hour_fraction` como referencia para el AP.


In [ ]:
evaluate_test = False
test_source = "A+B"  # También admite "A" o "B".

if evaluate_test:
    config = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
    saved_model = keras.models.load_model(run_dir / "best.keras", compile=False)
    saved_scaler = Standardization(**config["standardization"])
    test = load_partition(DATA_DIR, "test", test_source)
    test_batches = WindowBatches(
        PatientHours(test, saved_scaler),
        config["hyperparameters"]["window"],
        config["hyperparameters"].get("batch_size", config.get("batch_size")),
    )
    test_predictions = predict_hours(saved_model, test_batches)
    test_metrics = summarize_metrics(test_predictions, config["threshold"])
    test_predictions.to_csv(run_dir / f"test_{test_source}_predictions.csv", index=False)
    test_metrics.to_csv(run_dir / f"test_{test_source}_metrics.csv", index=False)
    display(test_metrics.round(4))
